# Pooled Analysis: All Datasets

> **Reproducibility note (read before running):** this notebook is one of a set of
> `0*_*.ipynb` notebooks split out of the original `compiled_results_analysis.ipynb`
> for the publication reproducibility bundle, meant to be run with the repo root (`task_dim/`) as
> the working directory. Each notebook is self-contained (re-loads its own
> inputs rather than relying on variables from other notebooks). A global `SEED = 42` is set at
> the top of the setup cell so every resampling/permutation step below is deterministic.
>
> **Inputs used here are pre-computed.** Where a cell loads a CSV/NIfTI file, a comment states
> which upstream script produced it. A few inputs this notebook depends on
> (`compiled/info/combined_participant_info.csv`, `compiled/results/*_compiled_results.csv`,
> `compiled/results/combined_corrs.csv`, `compiled/results/combined_corr_analyses.csv`) 

In [ ]:
import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import scipy
import scipy.stats as stats
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import ttest_rel, ttest_ind, wilcoxon, spearmanr
import nilearn
from nilearn import plotting, image, datasets
from nilearn.maskers import NiftiMasker
from nilearn.mass_univariate import permuted_ols
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.anova import anova_lm
import matplotlib.gridspec as gridspec
from patsy import dmatrix
from sklearn.utils import resample
import pingouin as pg
from matplotlib.patches import Patch
from matplotlib import rcParams
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

# Repo-local helper modules (see CLAUDE.md for the dataset-abstraction pattern)
import plotting_helpers as helper          # surface plots, colormaps, dataset colors
import stats_helpers as stats_helpers      # permutation tests, (parcelwise/LME) regression, FDR correction
import parcelwise_regressions as pwr       # run_regression_analyses, join_results_participant_info, clean_difference_dataframe
import hbn_config as hc                    # hc.AGE_BINS / hc.HBN_AGE_GROUPS used for HBN age binning

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

PLOT_DIR = 'compiled/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
ISC_CMAP = 'OrRd'
ID_CMAP = helper.custom_blues_cmap()
dataset_colors = helper.dataset_colors('all')

# Global seed: makes every unseeded resample/permutation call below reproducible.
SEED = 42
np.random.seed(SEED)

In [ ]:
# Common helper functions -- now defined once in plotting_helpers.py / stats_helpers.py
# (previously redefined identically in every final_results notebook; see REPRODUCIBILITY.md).
from plotting_helpers import (
    expand_parcellation_to_volume,  # parcel-array -> 3D volume
    load_atlas, get_region_order,   # Schaefer-400 atlas + region label order
    get_average_results, reorder_region_names,  # average/reindex per-region scores
)
from stats_helpers import lme_fit_summary  # MixedLM fit diagnostics (R^2, ICC, AIC/BIC)

## Overview: age / N distributions across all 5 datasets

In [ ]:
# INPUT: compiled/info/combined_participant_info.csv (see REPRODUCIBILITY.md)
par_df = pd.read_csv('compiled/info/combined_participant_info.csv')

_name_map = {'AdultRestMovie': 'adult_restmovie', 'HBN': 'hbn', 'InfantRestMovie': 'infant_restmovie',
             'Narratives': 'narratives', 'PartlyCloudy': 'partlycloudy'}
_label_map = {'infant_restmovie': 'Infant Rest/Movie', 'partlycloudy': 'Partly Cloudy',
              'hbn': 'Healthy Brain Network', 'adult_restmovie': 'Adult Rest/Movie', 'narratives': 'Narratives'}

par_df['dataset_key'] = par_df['dataset'].map(_name_map)
par_df['age_years'] = par_df['age_months'] / 12.0

_subs = par_df.drop_duplicates(subset=['participant_id', 'dataset_key'])[['participant_id', 'dataset_key', 'age_years']]
_order = (_subs.groupby('dataset_key')['age_years'].median().sort_values().index.tolist())[::-1]

_mo_vals = np.arange(0, 30, 4) / 12.0
_yr_vals = np.arange(5, 56, 5, dtype=float)
_all_ticks = np.concatenate([_mo_vals, _yr_vals])

def _fmt(t):
    m = round(t * 12)
    if m == 0: return '0'
    if m < 25: return ''
    return f'{int(round(t))}'

_tick_labels = [_fmt(t) for t in _all_ticks]
_x0, _x1 = 0.0, _subs['age_years'].max() + 1
_xgr = np.linspace(_x0, _x1, 1200)

n_ds = len(_order)
fig, axes = plt.subplots(n_ds, 1, figsize=(8, n_ds * 1.5), sharex=True, facecolor='white')
fig.subplots_adjust(hspace=0.35)

for i, (key, ax) in enumerate(zip(_order, axes)):
    ages = _subs.loc[_subs['dataset_key'] == key, 'age_years'].dropna().values
    n_sub = len(ages); lo, hi = ages.min(), ages.max()
    color = dataset_colors[key]
    kde = gaussian_kde(ages, bw_method='scott')
    ynorm = kde(_xgr) / kde(_xgr).max()
    ax.fill_between(_xgr, 0, ynorm, color=color, alpha=0.80, linewidth=0, zorder=2)
    ax.plot(_xgr, ynorm, color='black', linewidth=1.2, zorder=3)
    ax.axhline(0, color='black', linewidth=0.8, zorder=1)
    ax.set_ylim(-0.05, 1.2); ax.set_yticks([0.0, 0.5, 1.0])
    ax.set_yticklabels(['0', '0.5', '1'], fontsize=8, color='black')
    ax.spines['left'].set_color('black'); ax.spines['left'].set_linewidth(1)
    if i == n_ds - 1:
        ax.spines['bottom'].set_color('black'); ax.spines['bottom'].set_linewidth(1.0)
        ax.tick_params(axis='x', which='both', bottom=True, labelbottom=True, colors='black', direction='out', length=4, labelsize=8)
    else:
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.text(0.9, .95, _label_map[key], transform=ax.transAxes, ha='right', va='top', fontsize=16, fontweight='bold', color=color, zorder=5)
    ax.text(0.9, .74, f'N={n_sub},   {lo:.1f}\u2013{hi:.1f} yrs', transform=ax.transAxes, ha='right', va='top', fontsize=12, color='#333333', zorder=5)

axes[-1].set_xticks(_all_ticks)
axes[-1].set_xticklabels(_tick_labels, fontsize=12)
axes[-1].set_xlim(_x0, _x1)
fig.text(0.01, 0.5, 'Normalised density', va='center', ha='center', rotation='vertical', fontsize=12, color='black')
axes[0].set_title('Dataset age distributions', fontsize=16, pad=16)
fig.tight_layout()
plt.savefig('compiled/plots/age_distributions_ridge.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.show()

## Correspondence between ISC and ID by age (subject-level, all 5 datasets)

In [ ]:
# INPUT: compiled/results/combined_corrs.csv (see REPRODUCIBILITY.md)
df1 = pd.read_csv('compiled/results/combined_corrs.csv')

def get_sex(subject_id):
    participant_info = par_df[(par_df['participant_id'] == subject_id)]
    if not participant_info.empty:
        return participant_info.sex.values[0]

df1['sex'] = df1['participant_id'].apply(get_sex)
df1['FD'].fillna(df1['FD'].mean(), inplace=True)
df1['dataset'] = df1['dataset'].replace({'AdultRestMovie': 'adult_restmovie', 'HBN': 'hbn', 'Narratives': 'narratives',
                                          'InfantRestMovie': 'infant_restmovie', 'PartlyCloudy': 'partlycloudy'})

### Compare linear, quadratic, and log-age models (AIC/BIC/LRT)

In [ ]:
df1['log_age'] = np.log(df1['age_months'] + 1)

model_log_ml = smf.mixedlm('zscore ~ log_age + FD + sex', data=df1, groups=df1['dataset']).fit(reml=False)
model_lme1_ml = smf.mixedlm('zscore ~ age_months + FD + sex', data=df1, groups=df1['dataset']).fit(reml=False)

df1['age_months_sq'] = df1['age_months'] ** 2
df1['age_c'] = df1['age_months'] - df1['age_months'].mean()  # center age to reduce multicollinearity with age^2
df1['age_c_sq'] = df1['age_c'] ** 2
model_quad_ml = smf.mixedlm('zscore ~ age_c + age_c_sq + FD + sex', data=df1, groups=df1['dataset']).fit(reml=False)

results = pd.DataFrame({
    'model': ['linear', 'quadratic', 'log'],
    'AIC': [model_lme1_ml.aic, model_quad_ml.aic, model_log_ml.aic],
    'BIC': [model_lme1_ml.bic, model_quad_ml.bic, model_log_ml.bic],
    'loglik': [model_lme1_ml.llf, model_quad_ml.llf, model_log_ml.llf]
})

lrt_stat = 2 * (model_quad_ml.llf - model_lme1_ml.llf)
p_lrt = stats.chi2.sf(lrt_stat, df=1)
print(f"LRT linear vs quadratic: chi2(1) = {lrt_stat:.3f}, p = {p_lrt:.4f}")

best_aic = results['AIC'].min(); best_bic = results['BIC'].min()
results['delta_AIC'] = results['AIC'] - best_aic
results['delta_BIC'] = results['BIC'] - best_bic
print("\nModel comparison with delta AIC/BIC:")
print(results.sort_values('AIC'))

In [ ]:
lme_fit_summary(model_log_ml)

### Log-age model: fitted line + bootstrap CI band, all 5 datasets

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
color_dict = helper.dataset_colors('all')
hue_order = list(color_dict.keys())
color_palette = [color_dict[i] for i in hue_order]

age_range = np.linspace(df1['age_months'].min(), df1['age_months'].max(), 300)
pred_df = pd.DataFrame({'age_months': age_range, 'log_age': np.log(age_range + 1),
                         'FD': df1['FD'].mean(), 'sex': df1['sex'].mode()[0], 'dataset': df1['dataset'].mode()[0]})

def predict_fixed_only(model, pred_df):
    """Predict using fixed effects only, ignoring random effects."""
    X = dmatrix(model.model.data.design_info, pred_df, return_type='matrix')
    fe_names = model.fe_params.index.tolist()
    X_df = pd.DataFrame(np.array(X), columns=model.model.data.design_info.column_names)
    return X_df[fe_names].values @ model.fe_params.values

n_boot = 1000  # bootstrap resamples for the CI band; seeded per-iteration via random_state=i below
boot_preds = np.zeros((n_boot, len(age_range)))
print("Running bootstrap...")
for i in range(n_boot):
    if i % 100 == 0:
        print(f"  {i}/{n_boot}")
    boot_dfs = [resample(df1[df1['dataset'] == ds], replace=True, random_state=i) for ds in df1['dataset'].unique()]
    df_boot = pd.concat(boot_dfs).reset_index(drop=True)
    try:
        m_boot = smf.mixedlm('zscore ~ log_age + FD + sex', data=df_boot, groups=df_boot['dataset']).fit(reml=False, disp=False)
        boot_preds[i] = predict_fixed_only(m_boot, pred_df)
    except Exception as e:
        print(f"Boot {i} failed: {e}")
        boot_preds[i] = np.nan
print("Bootstrap complete.")

ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)
y_pred = model_log_ml.predict(pred_df)

ax.fill_between(age_range, ci_lower, ci_upper, color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred, color='gray', linewidth=1.5, zorder=3)
sns.scatterplot(x='age_months', y='zscore', data=df1, hue='dataset', hue_order=hue_order,
                palette=color_palette, s=30, alpha=0.8, ax=ax, zorder=1)

beta_val = model_log_ml.fe_params['log_age']
p_val = model_log_ml.pvalues['log_age']
stars = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
ax.text(0.95, 0.76, f'\u03b2={beta_val:.3f}{stars}', transform=ax.transAxes, ha='right', va='top', fontsize=10)

ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Age (months)', fontsize=12); ax.set_ylabel('z-score', fontsize=12)
ax.set_title('Task ID-ISC relationship across development', fontsize=12)
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/isc_id_movie_developmental_trajectory_all_datasets_logmodel.pdf',
            format='pdf', transparent=True, bbox_inches='tight')
plt.show()

## Build the 5-dataset parcelwise table (ISC + IDE + age + motion + sex, movie tasks only)

In [ ]:
# INPUTS: compiled/results/{partlycloudy,hbn,infant_restmovie,narratives,adult_restmovie}_compiled_results.csv
#   (see REPRODUCIBILITY.md). Movie/task-viewing rows only (rest/sleep dropped) so all 5
#   datasets contribute one task-viewing row per subject x region.
def get_age(subject_id):
    info = par_df[par_df['participant_id'] == subject_id]
    return info.age_months.values[0] if not info.empty else np.nan

def get_fd(subject_id):
    info = par_df[par_df['participant_id'] == subject_id]
    if info.empty or 'movie_FD' not in info.columns:
        return np.nan
    return info['movie_FD'].values[0]

pivot_cols = ['subject_id', 'region_name', 'task', 'age_months', 'dataset']

pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')
pc_df = pc_df.pivot_table(index=pivot_cols, columns='measure', values='score').reset_index()

hbn_df = pd.read_csv('compiled/results/hbn_compiled_results.csv')
hbn_df = hbn_df[hbn_df['task'] != 'rest'].reset_index(drop=True)
hbn_df = hbn_df.pivot_table(index=pivot_cols, columns='measure', values='score').reset_index()

inf_df_pool = pd.read_csv('compiled/results/infant_restmovie_compiled_results.csv')
inf_df_pool = inf_df_pool[inf_df_pool['task'] != 'sleep'].reset_index(drop=True)
inf_df_pool = inf_df_pool.pivot_table(index=pivot_cols, columns='measure', values='score').reset_index()

nar_df = pd.read_csv('compiled/results/narratives_compiled_results.csv').groupby(
    ['subject_id', 'region_name', 'measure'])['score'].mean().reset_index()
nar_df['age_months'] = nar_df['subject_id'].apply(get_age)
nar_df['dataset'] = 'narratives'
nar_df['task'] = 'narratives'
nar_df = nar_df.pivot_table(index=pivot_cols, columns='measure', values='score').reset_index()

adu_df = pd.read_csv('compiled/results/adult_restmovie_compiled_results.csv')
adu_df = adu_df[adu_df['task'] != 'rest'].reset_index(drop=True)
adu_df = adu_df.pivot_table(index=pivot_cols, columns='measure', values='score').reset_index()

for d in (pc_df, hbn_df, nar_df, adu_df, inf_df_pool):
    d['sex'] = d['subject_id'].apply(get_sex)
    d['movie_FD'] = d['subject_id'].apply(get_fd)

combined_df = pd.concat([pc_df, hbn_df, nar_df, adu_df, inf_df_pool], ignore_index=True)
combined_df.to_csv('compiled/results/combined_ID_ISC_with_covariates.csv', index=False)
combined_df.head()

## Parcelwise regression: IDE ~ ISC + log(age) + motion + sex + (1|dataset)

In [ ]:
combined_df = pd.read_csv('compiled/results/combined_ID_ISC_with_covariates.csv')
REGION_ORDER = get_region_order()
combined_df['dataset'] = combined_df['dataset'].astype(str).astype('category')
combined_df['sex'] = combined_df['sex'].astype('category')
combined_df['log_age'] = np.log(combined_df['age_months'] + 1)
combined_df = combined_df.reset_index(drop=True)
combined_df = combined_df.dropna(subset=['TPHATE_DiffOp_IDE', 'ISC', 'log_age', 'movie_FD', 'sex'])

formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex + (1|dataset)"
a = stats_helpers.parcelwise_regression(combined_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                         formula=formula, region_order=REGION_ORDER,
                                         alpha=0.05, fdr_method='fdr_bh', lme=True)
a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    print(f"Var: {var}, surviving significant coefs: {np.sum(sig_mask)}; "
          f"negative: {np.sum(coef_masked < 0)}, positive: {np.sum(coef_masked > 0)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked, image_fn=f'{PLOT_DIR}/all_datasets_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var}- combined', threshold=None, mask_medial_wall=True
    )

In [ ]:
# Which resting-state networks show significant log_age effects?
sig_regions = a[a['sig_log_age_fdr'] == 1]['region_name'].values
directions = [-1 if s < 0 else 1 for s in a[a['sig_log_age_fdr'] == 1]['coef_log_age'].values]
networks = {}
for region, direction in zip(sig_regions, directions):
    network = helper.get_schaefer_network(region)
    networks[network] = networks.get(network, 0) + 1 * direction
networks = dict(sorted(networks.items(), key=lambda item: item[1], reverse=True))
print("Networks with significant log_age effects:")
print(networks)

## Combining across datasets: average ID/ISC ~ age (subject-level means, all 5 datasets)

In [ ]:
combined_df = combined_df[combined_df['task'] != 'mickey'].reset_index(drop=True)
combined_df['dataset'] = combined_df['dataset'].astype('category')
combined_df['sex'] = combined_df['sex'].astype('category')
combined_df['region_name'] = combined_df['region_name'].astype('category')

mean_df = combined_df.groupby(['subject_id', 'dataset', 'age_months', 'movie_FD', 'sex']).agg(
    {'ISC': 'mean', 'TPHATE_DiffOp_IDE': 'mean'}).reset_index()
mean_df['log_age'] = np.log(mean_df['age_months'] + 1)

lme_isc_combined = smf.mixedlm('ISC ~ log_age + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
lme_ide_combined = smf.mixedlm('TPHATE_DiffOp_IDE ~ log_age + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
print("Combined ISC regression results:"); print(lme_isc_combined.summary())
print("\nCombined IDE regression results:"); print(lme_ide_combined.summary())

In [ ]:
color_dict = helper.dataset_colors('all')
hue_order = list(color_dict.keys())
color_palette = [color_dict[i] for i in hue_order]

age_range = np.linspace(mean_df['age_months'].min(), mean_df['age_months'].max(), 300)

def predict_fixed_only(model, pred_df):
    X = dmatrix(model.model.data.design_info, pred_df, return_type='matrix')
    fe_names = model.fe_params.index.tolist()
    X_df = pd.DataFrame(np.array(X), columns=model.model.data.design_info.column_names)
    return X_df[fe_names].values @ model.fe_params.values

def bootstrap_predictions(orig_model, dataframe, formula, pred_df, n_boot=1000):
    """Bootstrap predictions using fixed effects only (deterministic via random_state=i)."""
    boot_preds = np.zeros((n_boot, len(pred_df)))
    for i in range(n_boot):
        if i % 100 == 0:
            print(f"  {i}/{n_boot}")
        boot_dfs = [resample(dataframe[dataframe['dataset'] == ds], replace=True, random_state=i)
                    for ds in dataframe['dataset'].unique()]
        df_boot = pd.concat(boot_dfs).reset_index(drop=True)
        m_boot = smf.mixedlm(formula, data=df_boot, groups=df_boot['dataset']).fit(reml=False, disp=False)
        boot_preds[i] = predict_fixed_only(m_boot, pred_df)
    ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
    ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)
    y_pred = orig_model.predict(pred_df)
    return y_pred, ci_lower, ci_upper

pred_df_isc = pd.DataFrame({'age_months': age_range, 'log_age': np.log(age_range + 1),
                            'movie_FD': mean_df['movie_FD'].mean(), 'sex': mean_df['sex'].mode()[0],
                            'dataset': mean_df['dataset'].mode()[0]})
pred_df_ide = pred_df_isc.copy()

y_pred_isc, ci_lower_isc, ci_upper_isc = bootstrap_predictions(
    lme_isc_combined, mean_df, 'ISC ~ log_age + movie_FD + C(sex)', pred_df_isc, n_boot=10000)
print("Bootstrap complete ISC")
y_pred_ide, ci_lower_ide, ci_upper_ide = bootstrap_predictions(
    lme_ide_combined, mean_df, 'TPHATE_DiffOp_IDE ~ log_age + movie_FD + C(sex)', pred_df_ide, n_boot=10000)
print("Bootstrap complete IDE")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plt.rcParams['font.family'] = 'Arial'

ax = axes[0]
sns.scatterplot(x='age_months', y='ISC', data=mean_df, hue='dataset', hue_order=hue_order,
                palette=color_palette, s=30, alpha=0.8, ax=ax, zorder=1)
ax.fill_between(age_range, ci_lower_isc, ci_upper_isc, color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred_isc, color='gray', linewidth=1.5, zorder=3)
beta_age_isc_combined = lme_isc_combined.params['log_age']
p_age_isc_combined = lme_isc_combined.pvalues['log_age']
sig_isc = helper.get_asterisks_pvalue(p_age_isc_combined)
ax.text(0.92, 0.84, f'\u03b2={beta_age_isc_combined:.3f}{sig_isc}', transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (months)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC predicted by age')

ax = axes[1]
sns.scatterplot(x='age_months', y='TPHATE_DiffOp_IDE', data=mean_df, hue='dataset', hue_order=hue_order,
                palette=color_palette, s=30, alpha=0.8, ax=ax, zorder=1)
ax.fill_between(age_range, ci_lower_ide, ci_upper_ide, color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred_ide, color='gray', linewidth=1.5, zorder=3)
beta_age_ide_combined = lme_ide_combined.params['log_age']
p_age_ide_combined = lme_ide_combined.pvalues['log_age']
sig_ide = helper.get_asterisks_pvalue(p_age_ide_combined)
ax.text(0.92, 0.44, f'\u03b2={beta_age_ide_combined:.3f}{sig_ide}', transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (months)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average IDE predicted by age')

sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/all_datasets_combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)
plt.show()

## Subsection: PartlyCloudy + HBN only (a 2-dataset developmental subset)

In [ ]:
# Reload PartlyCloudy per-subject ISC/IDE means (see 04_partlycloudy.ipynb "6-panel figure" cell
# for single_isc / single_ide construction) and HBN per-subject means (see 05_hbn.ipynb "Mean
# ISC/IDE/FD by age group" cell for single_isc_hbn / single_ide_hbn construction), reproduced here
# so this notebook is self-contained.
results_df_pc = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')

def get_age_groupings_pc(age):
    if age < 5: return 'U05'
    elif age < 6: return 'U06'
    elif age < 8.2: return 'U082'
    elif age < 13: return 'U13'
    else: return 'Adult'

par_df_pc = par_df[par_df['dataset'] == 'PartlyCloudy'].reset_index(drop=True)
results_df_pc['AgeGroup'] = [get_age_groupings_pc(a) for a in results_df_pc['_age_raw']]
par_df_pc['movie_FD'] = par_df_pc['movie_FD']

df_regional_isc = results_df_pc[(results_df_pc['task'] == 'pixar') & (results_df_pc['measure'] == 'ISC')].copy()
single_isc = df_regional_isc.groupby('subject_id').mean(numeric_only=True).reset_index()
single_isc['sex'] = single_isc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
df_regional_ide = results_df_pc[(results_df_pc['task'] == 'pixar') & (results_df_pc['measure'] == 'TPHATE_DiffOp_IDE')].copy()
single_ide = df_regional_ide.groupby('subject_id').mean(numeric_only=True).reset_index()
single_ide['sex'] = single_ide['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])

results_df_hbn_pool = pd.read_csv('compiled/results/hbn_compiled_results.csv')
par_df_hbn = pd.read_csv('HBN/basic_cohort_info.csv', index_col=1)

df_regional_isc_hbn = results_df_hbn_pool[(results_df_hbn_pool['task'] == 'movieTP') & (results_df_hbn_pool['measure'] == 'ISC')].copy()
single_isc_hbn = df_regional_isc_hbn.groupby('subject_id').mean(numeric_only=True).reset_index()
single_isc_hbn['sex'] = single_isc_hbn['subject_id'].map(par_df_hbn['sex'])
df_regional_ide_hbn = results_df_hbn_pool[(results_df_hbn_pool['task'] == 'movieTP') & (results_df_hbn_pool['measure'] == 'TPHATE_DiffOp_IDE')].copy()
single_ide_hbn = df_regional_ide_hbn.groupby('subject_id').mean(numeric_only=True).reset_index()
single_ide_hbn['sex'] = single_ide_hbn['subject_id'].map(par_df_hbn['sex'])

single_isc_pc = single_isc.copy(); single_isc_pc['dataset'] = 'PartlyCloudy'
single_ide_pc = single_ide.copy(); single_ide_pc['dataset'] = 'PartlyCloudy'
single_isc_hbn_copy = single_isc_hbn.copy(); single_isc_hbn_copy['dataset'] = 'HBN'
single_ide_hbn_copy = single_ide_hbn.copy(); single_ide_hbn_copy['dataset'] = 'HBN'

single_isc_combined = pd.concat([single_isc_pc, single_isc_hbn_copy], ignore_index=True)
single_ide_combined = pd.concat([single_ide_pc, single_ide_hbn_copy], ignore_index=True)
single_isc_combined['age_y'] = single_isc_combined['age_months'] / 12
single_ide_combined['age_y'] = single_ide_combined['age_months'] / 12
single_isc_combined['mean_FD'] = single_isc_combined['subject_id'].map(par_df_hbn['movie_FD']).fillna(
    single_isc_combined['subject_id'].map(par_df_pc.set_index('participant_id')['movie_FD']))
single_ide_combined['mean_FD'] = single_ide_combined['subject_id'].map(par_df_hbn['movie_FD']).fillna(
    single_ide_combined['subject_id'].map(par_df_pc.set_index('participant_id')['movie_FD']))

lme_isc_combined_pc_hbn = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_isc_combined, groups=single_isc_combined['dataset']).fit()
lme_ide_combined_pc_hbn = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_ide_combined, groups=single_ide_combined['dataset']).fit()
print("PC+HBN combined ISC regression results:"); print(lme_isc_combined_pc_hbn.summary())
print("\nPC+HBN combined IDE regression results:"); print(lme_ide_combined_pc_hbn.summary())

beta_isc, p_isc = lme_isc_combined_pc_hbn.params['age_y'], lme_isc_combined_pc_hbn.pvalues['age_y']
ci_isc = lme_isc_combined_pc_hbn.conf_int().loc['age_y'].values
z_isc = lme_isc_combined_pc_hbn.tvalues['age_y']
beta_ide, p_ide = lme_ide_combined_pc_hbn.params['age_y'], lme_ide_combined_pc_hbn.pvalues['age_y']
ci_ide = lme_ide_combined_pc_hbn.conf_int().loc['age_y'].values
z_ide = lme_ide_combined_pc_hbn.tvalues['age_y']
print(f'ISC regression: \u03b2={beta_isc:.2e}, 95% CI=({ci_isc[0]:.2e}, {ci_isc[1]:.2e}), z={z_isc:.2f}, '
      f'p={p_isc:.2e}, {helper.get_asterisks_pvalue(p_isc)}')
print(f'IDE regression: \u03b2={beta_ide:.2e}, 95% CI=({ci_ide[0]:.2e}, {ci_ide[1]:.2e}), z={z_ide:.2f}, '
      f'p={p_ide:.2e}, {helper.get_asterisks_pvalue(p_ide)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dataset_palette = {'PartlyCloudy': helper.dataset_colors('partlycloudy'), 'HBN': helper.dataset_colors('hbn')}

ax = axes[0]
sns.scatterplot(data=single_isc_combined, x='age_y', y='score', hue='dataset', palette=dataset_palette,
                s=24, ax=ax, legend=True, edgecolor='k', linewidth=0.1)
sns.regplot(data=single_isc_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray', line_kws={'linewidth': 2})
rho_isc, p_rho_isc = stats.spearmanr(single_isc_combined['age_y'], single_isc_combined['score'])
sig_isc = helper.get_asterisks_pvalue(p_isc)
ax.text(0.92, 0.64, f'\u03b2={beta_isc:.3f}{sig_isc}\n\u03c1={rho_isc:.2f}', transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC ~ age + FD + (1|dataset)')

ax = axes[1]
sns.scatterplot(data=single_ide_combined, x='age_y', y='score', hue='dataset', palette=dataset_palette,
                s=24, ax=ax, legend=True, edgecolor='k', linewidth=0.1)
sns.regplot(data=single_ide_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray', line_kws={'linewidth': 2})
rho_ide, p_rho_ide = stats.spearmanr(single_ide_combined['age_y'], single_ide_combined['score'])
sig_ide = helper.get_asterisks_pvalue(p_ide)
ax.text(0.98, 0.4, f'\u03b2={beta_ide:.3f}{sig_ide}\n\u03c1={rho_ide:.2f}', transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average ID ~ age + FD + (1|dataset)')

sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)
plt.show()

### PC+HBN: parcelwise regressions (IDE ~ ISC + log_age + motion + sex), pooled and per-dataset

In [ ]:
pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')
hbn_df = pd.read_csv('compiled/results/hbn_compiled_results.csv')
hbn_df = hbn_df[hbn_df['task'] != 'rest'].reset_index(drop=True)

pc_df = pc_df.pivot_table(index=['subject_id', 'region_name', 'task', 'age_months', 'dataset'], columns='measure', values='score').reset_index()
hbn_df = hbn_df.pivot_table(index=['subject_id', 'region_name', 'task', 'age_months', 'dataset'], columns='measure', values='score').reset_index()

pc_df['sex'] = pc_df['subject_id'].apply(get_sex)
pc_df['movie_FD'] = pc_df['subject_id'].apply(get_fd)
hbn_df['sex'] = hbn_df['subject_id'].apply(get_sex)
hbn_df['movie_FD'] = hbn_df['subject_id'].apply(get_fd)

pc_hbn_df = pd.concat([pc_df, hbn_df], ignore_index=True)
pc_hbn_df['log_age'] = np.log(pc_hbn_df['age_months'] + 1)
REGION_ORDER = get_region_order()
formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex + (1|dataset)"
a = stats_helpers.parcelwise_regression(pc_hbn_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                         formula=formula, region_order=REGION_ORDER,
                                         alpha=0.05, fdr_method='fdr_bh', lme=True)
a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked, image_fn=f'{PLOT_DIR}/HBN_PC_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} HBN + PC log', threshold=None, mask_medial_wall=True
    )

In [ ]:
# Per-dataset (non-LME) versions, for comparison against the pooled PC+HBN model above
pc_df['log_age'] = np.log(pc_df['age_months'] + 1)
formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex"
a_pc = stats_helpers.parcelwise_regression(pc_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                            formula=formula, region_order=REGION_ORDER,
                                            alpha=0.05, fdr_method='fdr_bh', lme=False)
a_pc = a_pc.set_index('region_name').reindex(REGION_ORDER).reset_index()
for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a_pc[f'sig_{var}_fdr'].values
    coef_masked = a_pc[f'coef_{var}'].values * sig_mask
    print(f"PartlyCloudy -- Var: {var}, surviving significant coefs: {np.sum(sig_mask)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked, image_fn=f'{PLOT_DIR}/PartlyCloudy_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} Partly Cloudy', threshold=None, mask_medial_wall=True
    )

hbn_df['log_age'] = np.log(hbn_df['age_months'] + 1)
a_hbn = stats_helpers.parcelwise_regression(hbn_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula, region_order=REGION_ORDER,
                                             alpha=0.05, fdr_method='fdr_bh', lme=False)
a_hbn = a_hbn.set_index('region_name').reindex(REGION_ORDER).reset_index()
for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a_hbn[f'sig_{var}_fdr'].values
    coef_masked = a_hbn[f'coef_{var}'].values * sig_mask
    print(f"HBN -- Var: {var}, surviving significant coefs: {np.sum(sig_mask)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked, image_fn=f'{PLOT_DIR}/HBN_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} HBN', threshold=None, mask_medial_wall=True
    )